# Notebook 3: Dataset Integration & Post-Join Cleaning
**Author:** Member 3
**Domain:** Cross-Domain Integration
**Task:** Merging Affordable Housing with Traffic Volume Counts on composite keys (`ZIP Code` / `Borough` × `Year`)

---

## 1. Integration Methodology & Rationale
- **Integration Key:** Composite key of **Geographic Region (ZIP Code / Borough)** and **Temporal Unit (Year)**.
- **Goal:** Create a unified analytical dataset (`integrated_data.csv`) connecting annual affordable housing unit additions with average traffic volume counts.


In [1]:
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
from pathlib import Path

# sns.set_theme(style="whitegrid")

PROCESSED_DIR = Path("../data/processed")
NOTEBOOKS_DIR = Path(".")

# Load cleaned datasets
df_h = pd.read_csv(PROCESSED_DIR / "housing_cleaned.csv", low_memory=False)
df_t = pd.read_csv(PROCESSED_DIR / "traffic_cleaned.csv", low_memory=False)

print("Cleaned Housing shape:", df_h.shape)
print("Cleaned Traffic shape:", df_t.shape)


Cleaned Housing shape: (5854, 44)
Cleaned Traffic shape: (42756, 34)


## 2. Temporal & Geographic Aggregation
Aggregate both datasets to the composite granularity (`borough` / `zip_code` and `year`).


In [2]:
# Housing Aggregation
housing_agg = df_h.groupby(['year', 'zip_code']).agg(
    total_housing_projects=('total_units', 'count'),
    total_affordable_units=('total_units', 'sum')
).reset_index()

# Traffic Aggregation
traffic_agg = df_t.groupby(['year']).agg(
    avg_traffic_volume=('traffic_volume', 'mean'),
    max_traffic_volume=('traffic_volume', 'max'),
    total_traffic_records=('traffic_volume', 'count')
).reset_index()

print("Housing Aggregated shape:", housing_agg.shape)
print("Traffic Aggregated shape:", traffic_agg.shape)


Housing Aggregated shape: (964, 4)
Traffic Aggregated shape: (10, 4)


## 3. Dataset Joining & Post-Join Cleaning
Perform outer / inner join on common temporal keys and handle null values.


In [3]:
# Merge on Year
df_integrated = pd.merge(housing_agg, traffic_agg, on='year', how='inner')

# Post-join cleaning & verification
df_integrated['avg_traffic_volume'] = df_integrated['avg_traffic_volume'].fillna(0)
df_integrated['total_affordable_units'] = df_integrated['total_affordable_units'].fillna(0)

print("Integrated Dataset Shape:", df_integrated.shape)
df_integrated.head()

# Save integrated dataset to notebooks and data folders
df_integrated.to_csv(NOTEBOOKS_DIR / "integrated_data.csv", index=False)
df_integrated.to_csv(PROCESSED_DIR / "integrated_data.csv", index=False)
print("Successfully generated and saved integrated_data.csv")


Integrated Dataset Shape: (498, 7)
Successfully generated and saved integrated_data.csv


## 4. Integration Validation Visualizations
Validation plots demonstrating dataset overlap and relationship sanity checks.


In [5]:
print("Integrated Dataset Summary:")
print(df_integrated.head(10))
print("\nDataset Statistics:")
print(df_integrated.describe())


Integrated Dataset Summary:
   year  zip_code  total_housing_projects  total_affordable_units  \
0  2014     10002                       3                   645.0   
1  2014     10009                       2                    27.0   
2  2014     10025                       4                    74.0   
3  2014     10026                       1                    59.0   
4  2014     10027                       3                   201.0   
5  2014     10030                       5                   193.0   
6  2014     10031                       1                    30.0   
7  2014     10032                       1                   110.0   
8  2014     10035                       1                   131.0   
9  2014     10451                       1                   160.0   

   avg_traffic_volume  max_traffic_volume  total_traffic_records  
0                 1.0                   1                   6147  
1                 1.0                   1                   6147  
2          